# Test pruning constraints

## Librairies

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os

# Go to project root (adjust depth if needed)
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA_PATH = os.path.join(ROOT, "data")
sys.path.append(ROOT)

print("Added to path:", ROOT)

import torch
from torchvision import datasets, transforms
from torch.utils.data import Subset

from data.mnist_data import load_mnist_datasets
from src.models.networks import FashionMLP_Large
from src.quantization.quantize import quantize_model
from src.shortcuts.shortcut_weights import *
from src.optim.build_polytopes import *
from src.optim.prune_constraints import *

Added to path: /Users/jeremiecabessa/Desktop/ROOT/Articles/Conference_Papers/2025_With_Jiri/code/ErrorVolumePolytopes


In [2]:
torch.manual_seed(42)

## Large MLP (FashionMNIST)

In [3]:
# -------------------- #
# Load models and data #
# -------------------- #

# Device
device = torch.device("gpu" if torch.cuda.is_available() 
                      else "mps" if torch.backends.mps.is_available() 
                      else "cpu")

# Model
model_name = "fashion_mlp_best.pth"
model_path = os.path.join(ROOT, "checkpoints", model_name)

model = FashionMLP_Large()
model.load_state_dict(torch.load(model_path))#['model_state'])
model.to(device).eval()

# qModel
qmodel = quantize_model(model, bits=4)
qmodel.to(device).eval()

# Dataset (and sample)
train_dataset = torch.load(
    DATA_PATH + "/fashionMNIST_correct_mlp.pt",
    weights_only=False
)

x_0, c = train_dataset[123]
print(x_0.min().item(), x_0.max().item()) # check bounds
x_0 = x_0.flatten().unsqueeze(0) # shape (1, input_dim)
x_0 = x_0.to(device) # important
print("Sample x_0 shape:", x_0.shape)
print("Models and dataset have been loaded.")

-1.0 1.0
Sample x_0 shape: torch.Size([1, 784])
Models and dataset have been loaded.


In [ ]:
# ------------------------ #
# Compute correct polytope #
# ------------------------ #
print("*** Testing single sample in the polytopes... ***\n\n")

A_correct, b_correct, A_both, b_both = build_two_class_polytopes(model, qmodel, x_0, c)

print("\nBefore pruning:")
print("A_correct shape:", A_correct.shape)
print("b_correct shape:", b_correct.shape)

# print("\nPruning constraints with Clarkson method...")
# bounds = (-0.5, 2.9) # bounds for pixel values (after flattening and transformation)
# A_pruned, b_pruned, ratio_removed = prune_constraints_Clarkson(A_correct, 
#                                                         b_correct, 
#                                                         bounds=bounds, 
#                                                         verbose=True)
# print("\After pruning:")
# print("A_pruned shape:", A_pruned.shape)
# print("b_pruned shape:", b_pruned.shape)
# print("Ratio of constraints removed:", ratio_removed)


print("\nPruning constraints with Ray-Tracing + Clarkson method...")
bounds = (-1., 1.) # bounds for pixel values (after transformation)
A_pruned, b_pruned, ratio_removed = prune_constraints_RayTracing(A_correct, 
                                                                 b_correct, 
                                                                 bounds=bounds, 
                                                                 n_rays=10000,
                                                                 direction_mode="gaussian", # gaussian or sign or axis
                                                                 verbose=True)
print("\nAfter pruning:")
print("A_pruned shape:", A_pruned.shape)
print("b_pruned shape:", b_pruned.shape)
print("Ratio of constraints removed:", ratio_removed)